# Kaggle Homework — Tyler Wolf Williams — tylerwolfwilliams2

**Competition:** [Playground Series S6E4 — Predicting Irrigation Need](https://www.kaggle.com/competitions/playground-series-s6e4)  
**Kaggle Profile:** [tylerwolfwilliams2](https://www.kaggle.com/tylerwolfwilliams2)  
**Course:** Advanced Machine Learning  
**Date:** April 2026

---
## 1. Upvoted Resources

Below are 3–4 Kaggle notebooks and discussion posts I found particularly insightful and plan to apply to my modeling.

### Resource 1
**URL:** https://www.kaggle.com/code/saamhm/eda-baseline-model-fe-cv-ensemble-blend  
**Why I chose it:** This notebook covers the full pipeline end-to-end — EDA, feature engineering, cross-validation, ensembling, and blending. The ensemble/blend section is directly applicable to my Phase 2 plan: combining a bagging model (Random Forest) with a boosting model (LightGBM) via soft-vote blending is a concrete technique I can apply to squeeze out extra accuracy beyond either model alone. The CV methodology also gives me a solid benchmark to validate my own setup against.

### Resource 2
**URL:** https://www.kaggle.com/code/aryankaisth/s6e4-the-most-detailed-eda-100  
**Why I chose it:** Billed as the most detailed EDA for this competition, this notebook is useful for validating and expanding on my own EDA findings. A thorough EDA from the community helps me spot things I may have missed — e.g., unusual distributions, class separation patterns, or interactions between features like `Season` and `Crop_Type` that aren't obvious from looking at features individually. It's a good sanity check before committing to a modeling approach.

### Resource 3
**URL:** https://www.kaggle.com/code/cartoonistbeard/ps6e4-eda-baseline-cv-0-974  
**Why I chose it:** This notebook achieves a CV score of **0.974** with a baseline model, which sets a concrete performance target for my own work. Understanding how they reached that score — what preprocessing, model, and CV strategy they used — gives me a calibrated goal and a reproducible baseline to beat. If my tuned LightGBM falls significantly short, this notebook helps me diagnose where I'm leaving performance on the table.

### Resource 4
**URL:** https://www.kaggle.com/code/sakuno/irrigation-need-eda-orig-fe-cat  
**Why I chose it:** The "Orig FE | CAT" title signals this notebook does original feature engineering and uses CatBoost — two things directly on my Phase 2 roadmap. CatBoost handles categorical features natively without ordinal encoding, which could improve performance over my current LightGBM approach. The feature engineering section may also reveal domain-specific interactions (e.g., `Soil_Moisture × Rainfall_mm`) that I can borrow or adapt for my own notebooks.

---
## 2. Notebook Links

| Notebook | Location |
|---|---|
| EDA | [EDA.ipynb](./EDA.ipynb) |
| Bagging (Random Forest) | [Modeling_Bagging_RandomForest.ipynb](./Modeling_Bagging_RandomForest.ipynb) |
| Boosting (LightGBM) | [Modeling_Boosting_LightGBM.ipynb](./Modeling_Boosting_LightGBM.ipynb) |
| Kaggle EDA (public) | *(paste Kaggle notebook URL here if posted publicly)* |
| Kaggle Modeling (public) | *(paste Kaggle notebook URL here if posted publicly)* |

---
## 3. EDA Insights

**Key findings from the EDA notebook:**

- **Target distribution:** Imbalanced — Low: 369,917 (58.72%), Medium: 239,074 (37.95%), High: 21,009 (3.33%). The `High` class is severely underrepresented (~1/18th of `Low`). Class-weighting or oversampling should be considered.

- **Most important feature:** `Soil_Moisture` is by far the strongest predictor (ANOVA F = 82,556), followed by `Wind_Speed_kmh` (F = 22,514) and `Temperature_C` (F = 22,044). `Rainfall_mm` (F = 7,242) is also strong. `Sunlight_Hours` is the weakest numerical feature (F = 8.7).

- **Categorical features:** 8 categorical variables — `Soil_Type` (4 values), `Crop_Type` (6), `Crop_Growth_Stage` (4), `Season` (3: Kharif/Rabi/Zaid), `Irrigation_Type` (4), `Water_Source` (4), `Mulching_Used` (2), `Region` (5). `Season` encodes growing-season climate implicitly and shows meaningful variation across target classes.

- **Potential leakage concern:** `Irrigation_Type` (Drip, Sprinkler, Canal, Rainfed) and `Water_Source` are likely downstream of the label — worth testing models with and without both features.

- **Missing values:** None — train (630,000 rows) and test (270,000 rows) are fully populated across all features.

- **Train/test distribution:** No covariate shift detected — distributions are well-aligned across all numerical features.

- **New technique tried:** Plotted target-conditional KDE and box plots for all numeric features to visually identify class separability, and computed one-way ANOVA F-statistics to rank features by discriminative power.


---
## 4. Modeling Discussion

### Approach Summary

Both models use the same preprocessing pipeline:
- Ordinal encoding for all categorical features (RF); native `category` dtype for LightGBM
- No scaling needed (tree-based models are scale-invariant)
- Stratified 3-fold cross-validation
- Hyperparameter search via `RandomizedSearchCV` (8 iterations) on an 80K-row subsample
- Evaluation metric: **Accuracy** (competition metric) and macro-averaged F1

### Results Table

| Model | CV Accuracy (mean ± std) | CV Macro F1 |
|---|---|---|
| Random Forest (baseline) | 0.9852 ± 0.0001 | 0.9695 ± 0.0003 |
| Random Forest (tuned) | 0.9855 ± 0.0001 | 0.9710 ± 0.0001 |
| LightGBM (baseline) | 0.9847 ± 0.0001 | 0.9699 ± 0.0003 |
| LightGBM (tuned) | 0.9847 ± 0.0001 | 0.9701 ± 0.0002 |

### What Worked Well
- Both models achieved ~98.5% CV accuracy with minimal tuning, indicating the features are highly predictive.
- `Crop_Growth_Stage` and `Soil_Moisture` dominated RF feature importances (32% and 26%); LightGBM ranked `Rainfall_mm` and `Temperature_C` highest by split count.
- Tuning RF improved F1 Macro from 0.9695 to 0.9710 (+0.15%) with best params: n_estimators=100, max_depth=30, max_features=0.5.
- LightGBM’s early stopping cleanly identified 142 as the optimal number of trees, avoiding overfitting without manual tuning of `n_estimators`.
- Both models produced identical holdout accuracy (99.0%), suggesting the dataset is well-structured and learnable.

### What Didn't Work Well
- The `High` irrigation class (3.3% of data) had only 92% recall in both models — class imbalance is the main remaining weakness.
- LightGBM tuning gave essentially no improvement over the baseline (0.0000% accuracy gain, +0.02% F1), suggesting the baseline hyperparameters were already near-optimal.
- RF tuning also gave only marginal gains (+0.03% accuracy), confirming this dataset doesn’t require heavy tuning.

### Were Improvements Meaningful?
- RF tuning: +0.03% accuracy, +0.15% F1 Macro — consistent across folds (std tightened from 0.0003 to 0.0001) but practically small.
- LightGBM tuning: negligible accuracy change, +0.02% F1 — the baseline was effectively already tuned.
- The more meaningful metric is F1 Macro, since it captures performance on the minority `High` class.

### Boosting vs. Bagging
- RF tuned (0.9855 accuracy, 0.9710 F1) slightly outperformed LightGBM tuned (0.9847 accuracy, 0.9701 F1) on this dataset.
- The two models disagree on feature importance: RF assigns most weight to `Crop_Growth_Stage`, while LightGBM spreads importance more evenly across continuous features like `Rainfall_mm` and `Temperature_C`.
- RF was easier to tune and less sensitive to hyperparameter choices; LightGBM’s advantage is expected to emerge more with deeper tuning, feature engineering, or larger ensembles in Phase 2.


---
## 5. Phase 2 Plan

To improve model performance in the next iteration:

1. **Feature engineering:**
   - Create interaction terms: `Soil_Moisture × Rainfall_mm`, `Temperature_C × Humidity`
   - Bin `Rainfall_mm` into drought/normal/flood categories
   - Create a combined `climate_stress` index from temperature, humidity, and rainfall

2. **Model exploration:**
   - Try CatBoost (handles categoricals natively without encoding)
   - Try XGBoost with `tree_method='hist'` for speed
   - Explore stacking/blending of Random Forest + LightGBM + CatBoost

3. **Hyperparameter tuning:**
   - Use Optuna for Bayesian optimization on LightGBM (more efficient than RandomizedSearchCV)
   - Focus on `num_leaves`, `learning_rate`, `min_child_samples`, and `colsample_bytree`

4. **Validation strategy:**
   - Investigate whether a stratified split by `Region` + `Season` better reflects test set distribution

5. **Leakage investigation:**
   - Test model performance with and without `Irrigation_Type` and `Water_Source` to quantify leakage vs. signal